# Agent Running Test 
### Prerequisite for LLM used in the agent example
1. Install Ollama from https://ollama.com
2. Start Ollama `ollama run qwen3:4b`


### Prerequisites for a remote VM
1. Start an SSH server (e.g., `openssh-server`) so you can connect to the VM if needed.
2. Install Python virtual environment support (e.g., `python3-venv`).

In [1]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [1]:
example_path = 'agent.py'

## Python Interpreter

In [3]:
!python {example_path}

Using device: cuda
The tool detected the cat in the image with the following bounding boxes:

1. **Top-left**: (xmin=9, ymin=54, xmax=316, ymax=474)
2. **Top-right**: (xmin=346, ymin=23, xmax=639, ymax=372)

Let me know if you'd like further details!


## Run the agent and tool teogether locally

In [4]:
# Removed pip_install parameter from decorators in {example_path}
import ast 
import os
import tempfile



with open(example_path, "r") as f:
    source_code = f.read()
    mod = ast.parse(source_code)

# Remove env keyword argument
for node in ast.walk(mod):
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call):
                dec.keywords = [kw for kw in dec.keywords if kw.arg != "pip_install"]

with tempfile.NamedTemporaryFile("w", delete=False, suffix=".py") as temp_file:
    temp_file.write(ast.unparse(mod))
    temp_file_path = temp_file.name

!declagent run {temp_file_path}
os.unlink(temp_file_path)

Tool 'Grounding' assigned to Node 'local' 
Using device: cuda:0
The bounding boxes for the detected entity 'cat' are as follows:

1. **Box 1**:  
   - **xmin**: 9  
   - **ymin**: 54  
   - **xmax**: 316  
   - **ymax**: 474  

2. **Box 2**:  
   - **xmin**: 346  
   - **ymin**: 23  
   - **xmax**: 639  
   - **ymax**: 372  

These boxes represent the region where the 'cat' is identified in the image. Let me know if you need further details!


## Run the agent locally while executing the tool inside a local Python virtual environment.

In [8]:
!declagent run {example_path}

Tool 'Grounding' assigned to Node 'local' 
Installing torch from https://download.pytorch.org/whl/cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp313-cp313-win_amd64.whl.metadata (28 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/setuptools-70.2.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached MarkupSafe-2.1.5-py3-none-any.whl
Using cached https://download.pytorch.org/wh


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\projects\declagent\example\CIGEva\.declagent_cache\Grounding\.venv\Scripts\python.exe -m pip install --upgrade pip
INFO:     Started server process [51884]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:52273 (Press CTRL+C to quit)


## Run the agent locally while executing the tool inside a remote Python virtual environment.

In [2]:
!declagent run {example_path} -c "exec_nodes_remote.toml"

Tool 'Grounding' assigned to Node 'vm1' 

STARTING REMOTE DEPLOYMENT: Grounding
Connecting to 172.30.162.209:22 via SSH...
✓ Connected to 172.30.162.209:22 via SSH
Allocating remote port...
✓ Allocated remote port 41133 on 172.30.162.209
Writing code and preparing requirements...
Creating remote directory /tmp/.declagent_cache/Grounding...
Uploading files via SFTP...
✓ Copied code to remote directory /tmp/.declagent_cache/Grounding
Creating virtual environment...
✓ Created venv at /tmp/.declagent_cache/Grounding/.venv
Installing 1 special package(s)...
  Installing torch from https://download.pytorch.org/whl/cu124...
    ✓ Installed torch
Installing packages from requirements.txt...
✓ Installed all requirements
Starting server...
  Waiting for server to start...
  Testing connectivity from local machine to http://172.30.162.209:41133...
  ✓ Server is accessible from local machine at http://172.30.162.209:41133
Here are the detected bounding boxes:

1. **xmin**: 9, **ymin**: 54, **xmax*

## Clear Cache on nodes

In [7]:
!declagent clear